# Milestone 2: Baseline YOLO Fine-Tuning

This notebook creates a reproducible internal train/validation split from the original 512 training images, smoke-tests YOLO11n, trains the baseline, evaluates the **soccer-ball** class, and saves presentation-ready predictions. The original 100-image `images/val` partition is never used here; it remains the final held-out test set.

Run all cells from the repository root. Full training is deliberately gated on CUDA.

In [1]:
from collections import Counter
from pathlib import Path
import hashlib, importlib.util, json, random, shutil, sys, time

import numpy as np
import torch
import yaml
from PIL import Image

ROOT = Path.cwd().resolve()
assert (ROOT / 'data.yaml').exists(), 'Run this notebook from the repository root.'
RESULTS = ROOT / 'results' / 'baseline'
SPLITS = RESULTS / 'splits'
PREDICTIONS = RESULTS / 'predictions'
for folder in (RESULTS, SPLITS, PREDICTIONS): folder.mkdir(parents=True, exist_ok=True)
SEED, IMAGE_SIZE, MAX_EPOCHS, PATIENCE = 4620, 640, 50, 10
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

with open(ROOT / 'data.yaml', encoding='utf-8') as f:
    original_config = yaml.safe_load(f)
CLASS_NAMES = {int(k): str(v) for k, v in original_config['names'].items()}
ball_matches = [i for i, name in CLASS_NAMES.items() if name.strip().lower() in {'ball', 'soccer ball', 'football'}]
assert len(ball_matches) == 1, f'Expected exactly one ball class, found {ball_matches}'
BALL_CLASS_ID = ball_matches[0]  # discovered from data.yaml; never assumed
print('Classes:', CLASS_NAMES)
print('Soccer-ball class ID:', BALL_CLASS_ID)

Classes: {0: 'ball', 1: 'player', 2: 'referee'}
Soccer-ball class ID: 0


## 1. Validate paths and protect the held-out test set

Only `images/train` is split below. `images/val` is inventoried but never placed in a training YAML.

In [2]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
def image_files(folder):
    return sorted(p.resolve() for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

source_images = image_files(ROOT / 'images' / 'train')
heldout_images = image_files(ROOT / 'images' / 'val')
assert len(source_images) == 512 and len(heldout_images) == 100

def label_for(image_path):
    return ROOT / 'labels' / image_path.parent.name / f'{image_path.stem}.txt'

missing = [p for p in source_images + heldout_images if not label_for(p).exists()]
assert not missing, f'Missing labels: {missing[:5]}'
assert not ({p.name for p in source_images} & {p.name for p in heldout_images})
print(f'Internal split source: {len(source_images)} original training images')
print(f'Protected final test set: {len(heldout_images)} images (not used below)')

Internal split source: 512 original training images
Protected final test set: 100 images (not used below)


## 2. Leakage-aware, reproducible 80/20 split

Filenames reveal 15 capture-session prefixes, but two sessions contain 276 and 194 of the 512 images. Holding out an entire session therefore cannot produce an approximately 20% validation set. To reduce obvious leakage while preserving the requested split size, the code groups visually near-duplicate frames using a 64-bit difference hash (Hamming distance ≤ 6), then assigns whole visual groups with seed 4620. This prevents detected near-duplicates from crossing the split, although source-video leakage cannot be ruled out because frame order/video IDs are not fully available.

In [3]:
def source_prefix(path):
    parts = path.stem.rsplit('_', 1)
    return parts[0] if len(parts) == 2 and len(parts[1]) == 6 else path.stem

def difference_hash(path, size=8):
    with Image.open(path) as image:
        pixels = np.asarray(image.convert('L').resize((size + 1, size)))
    bits = (pixels[:, 1:] > pixels[:, :-1]).ravel()
    return sum(int(bit) << i for i, bit in enumerate(bits))

hashes = [difference_hash(p) for p in source_images]
parent = list(range(len(source_images)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def union(a, b):
    a, b = find(a), find(b)
    if a != b: parent[b] = a
for i in range(len(hashes)):
    for j in range(i):
        if (hashes[i] ^ hashes[j]).bit_count() <= 6: union(i, j)

visual_groups = {}
for i, path in enumerate(source_images): visual_groups.setdefault(find(i), []).append(path)
groups = list(visual_groups.values())
rng = random.Random(SEED); rng.shuffle(groups)
target_val = round(0.20 * len(source_images))
groups.sort(key=len, reverse=True)
internal_val, internal_train = [], []
for group in groups:
    if len(internal_val) + len(group) <= target_val:
        internal_val.extend(group)
    else:
        internal_train.extend(group)
assert len(internal_train) + len(internal_val) == 512
assert not (set(internal_train) & set(internal_val))
assert not (set(internal_train + internal_val) & set(heldout_images))
print(f'Train: {len(internal_train)} ({len(internal_train)/512:.1%})')
print(f'Internal validation: {len(internal_val)} ({len(internal_val)/512:.1%})')
print(f'Visual groups: {len(groups)}; largest: {max(map(len, groups))}')
print('Capture sessions:', dict(Counter(source_prefix(p) for p in source_images)))

Train: 410 (80.1%)
Internal validation: 102 (19.9%)
Visual groups: 503; largest: 2
Capture sessions: {'20250710_144746': 276, '20250710_145737': 194, '20250812_101950': 3, '20250812_101951': 14, '20250812_102022': 1, '20250812_102037': 1, '20250812_102038': 5, '20250812_102039': 4, '20250812_102508': 1, '20250812_102527': 1, '20250812_102528': 3, '20250812_102532': 1, '20250812_102533': 1, '20250812_103112': 1, '20250812_103121': 6}


In [4]:
train_manifest = SPLITS / 'train.txt'
val_manifest = SPLITS / 'val.txt'
train_manifest.write_text(''.join(p.as_posix() + '\n' for p in sorted(internal_train)), encoding='utf-8')
val_manifest.write_text(''.join(p.as_posix() + '\n' for p in sorted(internal_val)), encoding='utf-8')
internal_yaml = RESULTS / 'internal_data.yaml'
split_config = {'path': ROOT.as_posix(), 'train': train_manifest.as_posix(),
                'val': val_manifest.as_posix(), 'nc': len(CLASS_NAMES), 'names': CLASS_NAMES}
internal_yaml.write_text(yaml.safe_dump(split_config, sort_keys=False), encoding='utf-8')

def class_counts(paths):
    counts = Counter()
    for image_path in paths:
        for line in label_for(image_path).read_text().splitlines():
            if line.strip(): counts[int(line.split()[0])] += 1
    return {CLASS_NAMES[i]: counts[i] for i in sorted(CLASS_NAMES)}
print('Train annotations:', class_counts(internal_train))
print('Internal-val annotations:', class_counts(internal_val))
assert class_counts(internal_val)[CLASS_NAMES[BALL_CLASS_ID]] > 0

Train annotations: {'ball': 334, 'player': 5855, 'referee': 512}
Internal-val annotations: {'ball': 83, 'player': 1423, 'referee': 122}


## 3. Environment and GPU check

Install dependencies with `pip install -r requirements.txt`. The pretrained checkpoint may be downloaded by Ultralytics on first use. A CPU may be used for the tiny smoke test, but the 50-epoch run is blocked unless CUDA is available.

In [5]:
ULTRALYTICS_INSTALLED = importlib.util.find_spec('ultralytics') is not None
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = 0 if CUDA_AVAILABLE else 'cpu'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Ultralytics installed:', ULTRALYTICS_INSTALLED)
print('CUDA available:', CUDA_AVAILABLE)
print('GPU:', torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'None — full training will not run')

Python: 3.12.0
PyTorch: 2.15.0+cpu
Ultralytics installed: False
CUDA available: False
GPU: None — full training will not run


## 4. Baseline settings and default augmentations

The augmentation values are written explicitly rather than silently inherited. They reproduce the standard Ultralytics detection defaults and establish the control condition for the later augmentation experiment: HSV (`0.015/0.7/0.4`), translation `0.1`, scale `0.5`, horizontal flip `0.5`, mosaic `1.0`, and no rotation, shear, perspective, vertical flip, mixup, cutmix, or copy-paste. Mosaic closes for the last 10 epochs.

In [6]:
TRAIN_ARGS = dict(data=str(internal_yaml), imgsz=IMAGE_SIZE, epochs=MAX_EPOCHS,
    patience=PATIENCE, batch=16, device=DEVICE, workers=0, seed=SEED,
    deterministic=True, project=str(RESULTS), name='train', exist_ok=True,
    plots=True, save=True, verbose=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.5, bgr=0.0, mosaic=1.0, mixup=0.0,
    cutmix=0.0, copy_paste=0.0, close_mosaic=10)
run_config = {'model': 'yolo11n.pt', 'seed': SEED, 'ball_class_id': BALL_CLASS_ID,
              'class_names': CLASS_NAMES, 'train_args': TRAIN_ARGS}
(RESULTS / 'training_config.json').write_text(json.dumps(run_config, indent=2, default=str), encoding='utf-8')
print(json.dumps(run_config, indent=2, default=str))

{
  "model": "yolo11n.pt",
  "seed": 4620,
  "ball_class_id": 0,
  "class_names": {
    "0": "ball",
    "1": "player",
    "2": "referee"
  },
  "train_args": {
    "data": "C:\\Users\\javie\\OneDrive\\Documents\\data4620\\Football-Image-Dataset-main\\results\\baseline\\internal_data.yaml",
    "imgsz": 640,
    "epochs": 50,
    "patience": 10,
    "batch": 16,
    "device": "cpu",
    "workers": 0,
    "seed": 4620,
    "deterministic": true,
    "project": "C:\\Users\\javie\\OneDrive\\Documents\\data4620\\Football-Image-Dataset-main\\results\\baseline",
    "name": "train",
    "exist_ok": true,
    "plots": true,
    "save": true,
    "verbose": true,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "bgr": 0.0,
    "mosaic": 1.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 10
  }
}


## 5. One-epoch smoke test

This uses only 3% of the internal training data, a batch of two, and one epoch. It checks model loading, paths, labels, and the training loop before the real run.

In [7]:
SMOKE_TEST_OK = False
if not ULTRALYTICS_INSTALLED:
    print('SKIPPED: install Ultralytics with: pip install -r requirements.txt')
else:
    from ultralytics import YOLO, __version__ as ultralytics_version
    print('Ultralytics:', ultralytics_version)
    smoke_model = YOLO('yolo11n.pt')
    smoke_model.train(data=str(internal_yaml), imgsz=IMAGE_SIZE, epochs=1, fraction=0.03,
                      batch=2, device=DEVICE, workers=0, seed=SEED, deterministic=True,
                      project=str(RESULTS), name='smoke', exist_ok=True, plots=False)
    SMOKE_TEST_OK = True
    print('Smoke test passed.')

SKIPPED: install Ultralytics with: pip install -r requirements.txt


## 6. Full baseline training

After the smoke test passes on a CUDA machine, set `RUN_FULL_TRAIN = True`. Early stopping uses patience 10 and the best checkpoint is saved under `results/baseline/train/weights/best.pt`.

In [8]:
RUN_FULL_TRAIN = False  # change to True only after reviewing the smoke test
training_seconds = None
if RUN_FULL_TRAIN and CUDA_AVAILABLE and SMOKE_TEST_OK:
    model = YOLO('yolo11n.pt')
    started = time.perf_counter()
    model.train(**TRAIN_ARGS)
    training_seconds = time.perf_counter() - started
    (RESULTS / 'training_time.json').write_text(json.dumps({'seconds': training_seconds}, indent=2))
    print(f'Training completed in {training_seconds/60:.1f} minutes.')
elif not CUDA_AVAILABLE:
    print('NOT STARTED: CUDA is unavailable; refusing the long CPU training run.')
elif not SMOKE_TEST_OK:
    print('NOT STARTED: the smoke test must pass first.')
else:
    print('Ready. Set RUN_FULL_TRAIN=True to start the reviewed GPU run.')

NOT STARTED: CUDA is unavailable; refusing the long CPU training run.


## 7. Evaluate the best checkpoint on internal validation only

The ball row is found from `ap_class_index`, so its position is never confused with its class ID. The protected 100-image test partition is not referenced.

In [9]:
best_weights = RESULTS / 'train' / 'weights' / 'best.pt'
ball_metrics = None
if best_weights.exists() and ULTRALYTICS_INSTALLED:
    best_model = YOLO(str(best_weights))
    metrics = best_model.val(data=str(internal_yaml), split='val', imgsz=IMAGE_SIZE,
                             device=DEVICE, plots=True, project=str(RESULTS),
                             name='validation', exist_ok=True)
    evaluated_ids = list(map(int, metrics.box.ap_class_index))
    position = evaluated_ids.index(BALL_CLASS_ID)
    precision, recall, map50, map5095 = map(float, metrics.box.class_result(position))
    ball_metrics = {'class_id': BALL_CLASS_ID, 'class_name': CLASS_NAMES[BALL_CLASS_ID],
                    'precision': precision, 'recall': recall, 'mAP50': map50,
                    'mAP50-95': map5095}
    (RESULTS / 'ball_metrics.json').write_text(json.dumps(ball_metrics, indent=2))
    print(json.dumps(ball_metrics, indent=2))
else:
    print('SKIPPED: best.pt does not exist because full training has not run.')

SKIPPED: best.pt does not exist because full training has not run.


## 8. Save representative validation predictions

Predictions are matched to ball ground truth at IoU ≥ 0.5. The selection prioritizes true detections, missed balls, and false positives when those categories exist. Five annotated images and an error-summary JSON are saved. Common likely causes include the ball's tiny pixel size, blur/occlusion, and ball-like white field markings.

In [10]:
def ball_ground_truth(image_path):
    with Image.open(image_path) as im: w, h = im.size
    boxes = []
    for line in label_for(image_path).read_text().splitlines():
        if not line.strip(): continue
        cls, x, y, bw, bh = map(float, line.split())
        if int(cls) == BALL_CLASS_ID:
            boxes.append([(x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h])
    return np.asarray(boxes, dtype=float).reshape(-1, 4)

def iou_matrix(a, b):
    if not len(a) or not len(b): return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.maximum(br-tl, 0), axis=2)
    area_a = np.prod(a[:, 2:]-a[:, :2], axis=1)[:, None]
    area_b = np.prod(b[:, 2:]-b[:, :2], axis=1)[None, :]
    return inter / np.maximum(area_a + area_b - inter, 1e-9)

if best_weights.exists() and ULTRALYTICS_INSTALLED:
    records = []
    predictions = best_model.predict([str(p) for p in sorted(internal_val)], imgsz=IMAGE_SIZE,
                                     conf=0.25, device=DEVICE, verbose=False)
    for image_path, result in zip(sorted(internal_val), predictions):
        gt = ball_ground_truth(image_path)
        keep = result.boxes.cls.cpu().numpy().astype(int) == BALL_CLASS_ID
        pred = result.boxes.xyxy.cpu().numpy()[keep]
        overlaps = iou_matrix(gt, pred)
        tp = min(len(gt), len(pred), int((overlaps.max(axis=1) >= .5).sum()) if len(pred) else 0)
        record = {'image': image_path, 'result': result, 'tp': tp,
                  'fn': len(gt)-tp, 'fp': len(pred)-tp}
        record['category'] = 'missed_ball' if record['fn'] else ('false_positive' if record['fp'] else 'correct_detection')
        records.append(record)
    chosen = []
    for category in ('correct_detection', 'missed_ball', 'false_positive'):
        chosen.extend([r for r in records if r['category'] == category][:2])
    for record in records:
        if len(chosen) >= 5: break
        if record not in chosen: chosen.append(record)
    summaries = []
    for i, record in enumerate(chosen[:5], 1):
        output = PREDICTIONS / f'{i}_{record["category"]}_{record["image"].name}'
        record['result'].save(filename=str(output))
        summaries.append({k: (v.name if k == 'image' else v) for k, v in record.items() if k not in {'result'}})
    (PREDICTIONS / 'error_summary.json').write_text(json.dumps(summaries, indent=2), encoding='utf-8')
    print(json.dumps(summaries, indent=2))
else:
    print('SKIPPED: predictions require the trained best.pt checkpoint.')

SKIPPED: predictions require the trained best.pt checkpoint.


## Check-in summary

Report the elapsed time from `training_time.json`, the four ball metrics from `ball_metrics.json`, and examples from `predictions/`. Training curves and confusion matrices are produced by Ultralytics under `results/baseline/train` and `results/baseline/validation`. Do not use the original 100-image held-out set until the final evaluation.